In [ ]:
# 验证可编辑安装：__file__ 必须指向你自己项目里的 src/signal_insight/
import signal_insight
from pathlib import Path
print(signal_insight.__version__, signal_insight.__file__)
assert Path(signal_insight.__file__).parent.parent.name == "src", "装到别处去了：见《可编辑安装》第 10 节"
print("✅ 可编辑安装成功")

In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_mysql import database as mysql
from signal_insight import config, viz, quality
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT  # 配色常量

assert os.environ.get("SIGNAL_MYSQL_PASSWORD"),\
    "没读到 MySQL 密码：关掉 Jupyter，在 PowerShell 里先运行 . .\\.env.ps1，再在同一个窗口运行 jupyterlab"
assert mysql.schema_exists(config.DB_SCHEMA),\
    f"MySQL 里没有库 {config.DB_SCHEMA}：回到 D01-1 第 6 步，用 tools/import_sqlite.py 把 signal.db 迁进来"

viz.apply_theme()  # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)  # 库名从包里来：notebook 在哪个文件夹打开都一样
DAY = config.ANALYSIS_DAY  # 分析日期（口径常量）
NB = "D02_信令数据验收"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"  # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

In [ ]:
from signal_mysql import database as mysql
# 验证 quality.dq_record：重记 D01 的 DQ-01，同一编号覆盖，稽核表行数不变
n_before = mysql.read_frame("SELECT COUNT(*) AS n FROM dq_check_result", con)["n"][0]
quality.dq_record(con, "DQ-01", "时效性", "各日数据完整度", "ods_signal.timestamp", "10 月 3 日占 97.3%", "警告",
                  evidence="10-01 仅 54 条，10-02 为 1.4 万条零散记录", decision="分析范围限定为 2018-10-03（D2 执行）")
n_after = mysql.read_frame("SELECT COUNT(*) AS n FROM dq_check_result", con)["n"][0]
assert n_before == n_after, f"稽核表行数 {n_before} → {n_after}：同一 check_id 应该覆盖，不该多出一行"
print("✅ 封装验证通过：quality.dq_record 与 D01 的写法写进同一张表，同一编号覆盖")

In [ ]:
from signal_mysql import database as mysql
raw = mysql.read_frame("SELECT * FROM ods_signal", con)
raw["ts"] = pd.to_datetime(raw["timestamp"], unit="ms") + pd.Timedelta(hours=config.UTC_OFFSET_HOURS)   # D1 做过：转北京时间；+8 是口径，从 config 取
raw.shape

In [ ]:
from signal_mysql import database as mysql
# 这一格在做什么：统计每个字段缺多少。
# 注意「缺失」有两种：真空值（NaN / NULL），和**看起来有值、其实是空白字符串**（" "）。
# 只用 isna() 会漏掉后者——本数据的 tmp1 整列都是空格，isna() 一个都查不出来。
# 讲台提示：这是全课第一个“想当然就会出错”的点，值得停下来让学生自己试一次 isna()。
# drop(columns="ts") 是因为 ts 是我们自己算出来的，不属于原始字段，不参与质量检查。
missing = pd.DataFrame({
    "n_missing": raw.drop(columns="ts").isna().sum()
              + raw.drop(columns="ts").apply(lambda s: s.astype("string").str.strip().eq("").sum()
                                              if (pd.api.types.is_string_dtype(s) or s.dtype == object) else 0)
}).rename_axis("field").reset_index()
missing["missing_rate"] = missing["n_missing"] / len(raw)
missing = missing.sort_values("missing_rate").reset_index(drop=True)
mysql.write_frame(missing, "ads_missing_by_field", con, if_exists="replace", index=False)
missing

In [ ]:
from signal_mysql import database as mysql
m = mysql.read_frame("SELECT * FROM ads_missing_by_field", con)
full = m[m["missing_rate"] == 1]                               # 整列为空的字段
worst_other = m.loc[m["missing_rate"] < 1, "n_missing"].max()     # 其余字段里缺得最多的

fig, ax = plt.subplots(figsize=(10, 5.6),dpi=100)
colors = [RED if r == 1 else GRAY for r in m["missing_rate"]]  # 红色 = 整列不可用（警示）
bars = ax.barh(m["field"], m["missing_rate"], color=colors, height=0.62)
labels = ["100%（整列为空）" if r == 1 else (f"{n} 行" if n > 0 else "0") for r, n in zip(m["missing_rate"], m["n_missing"])]
ax.bar_label(bars, labels=labels, padding=4, color=INK)
ax.set_xlim(0, 1.25)
ax.set_xticks([])
ax.grid(False)
ax.spines["bottom"].set_visible(False)

title = f"{'、'.join(sorted(full['field']))} 整列为空，其余字段最多缺 {worst_other} 行"
viz.finish(fig, title,
           f"各字段缺失率；空值和空白字符串都算缺失；口径：ods_signal 全部 {len(raw):,} 行",
           SOURCE, "D2_01_字段缺失率.png")

In [ ]:
# 对账
assert set(full["field"]) == {"tmp0", "tmp1"} and worst_other == 15
print("✅ 对账通过：tmp0、tmp1 整列为空；其余字段最多缺 15 行（imsi）")

In [ ]:
quality.dq_record(con, "DQ-02", "完整性", "整列为空的字段", "ods_signal.tmp0 / tmp1", "2 个字段 100% 为空", "不通过",
          evidence="tmp0 全部为空值，tmp1 全部为空格", decision="删除这两个字段")
quality.dq_record(con, "DQ-03", "完整性", "关键字段缺失", "ods_signal.imsi / lac_id / cell_id",
          f"{raw[['imsi', 'lac_id', 'cell_id']].isna().any(axis=1).sum()} 行", "通过",
          evidence="imsi 缺 15、lac_id 缺 4、cell_id 缺 5，占比不到 0.01%", decision="删除这些行")

In [ ]:
# 这一格在做什么：检查 IMSI 的格式。IMSI 是手机卡的唯一标识，这份数据里应该是 18 位纯数字。
#   fullmatch   要求**整个字符串**都匹配。用 match 的话 "123abc" 也会算通过（只看开头）。
#   r"\d{18}"   恰好 18 个数字。
#   na=False    空值既不算匹配也不算不匹配，不写这个参数会直接报错——学生最常卡在这里。
ids = raw["imsi"].dropna()
bad_imsi = raw[raw["imsi"].notna() & ~raw["imsi"].str.fullmatch(r"\d{18}", na=False)]
print(f"IMSI 非空 {len(ids):,} 行，其中长度为 18 的 {ids.str.len().eq(18).sum():,} 行")
print(f"不是 18 位纯数字的：{len(bad_imsi)} 行")
bad_imsi[["imsi", "phone", "lac_id", "cell_id", "nid"]]

In [ ]:
phone_len = raw["phone"].str.len().value_counts()
odd_phone = raw.loc[raw["phone"].str.len() != 14, "phone"].value_counts()
print("手机号长度分布：", phone_len.to_dict())
print("异常手机号：", odd_phone.to_dict(), f"，涉及 {raw.loc[raw['phone'] == '6666', 'imsi'].nunique()} 个 IMSI")

In [ ]:
# 对账
assert len(bad_imsi) == 8 and odd_phone.to_dict() == {"6666": 36}
print("✅ 对账通过：8 条 IMSI 含掩码字符；36 条手机号是占位值 6666")

In [ ]:
quality.dq_record(con, "DQ-04", "规范性", "IMSI 格式", "ods_signal.imsi", f"{len(bad_imsi)} 行含 * ^ # 等掩码字符", "警告",
          evidence="长度都是 18，但不是纯数字，疑似脱敏时误处理", decision="删除这些行，并反馈数据提供方")
quality.dq_record(con, "DQ-05", "规范性", "手机号格式", "ods_signal.phone", "36 行为占位值 6666（14 个 IMSI）", "警告",
          evidence="其余手机号均为 14 位", decision="手机号不作为主键使用；分析中不需要手机号，明细表不保留")

In [ ]:
# 这一格在做什么：查「完全重复」——所有原始字段都一模一样的记录。
# 讲台提示：**清洗顺序会改变结果**，这是本格的重点。
# 这里先删缺失、再删格式异常、最后才判重复；如果先判重复，那些待删的脏行也会被算进重复里，
# 重复数就对不上了。企业里清洗步骤的顺序要写进文档，就是为了别人能复现出同一个数。
orig_cols = ["timestamp", "imsi", "lac_id", "cell_id", "phone", "timestamp1", "nid", "npid"]   # tmp0/tmp1 已决定删除
step = raw.dropna(subset=["imsi", "lac_id", "cell_id"])        # ① 关键字段缺了就没法定位，整行丢掉
step = step[step["imsi"].str.fullmatch(r"\d{18}")]             # ② 格式不对的 IMSI 认不了人
dup_mask = step.duplicated(subset=orig_cols, keep="first")     # ③ 每组重复只留第一条
print(f"完全重复：{dup_mask.sum():,} 行，占 {dup_mask.mean():.2%}")
step[step.duplicated(subset=orig_cols, keep=False)].sort_values(orig_cols).head(4)

In [ ]:
# 对账
assert dup_mask.sum() == 2_568
print("✅ 对账通过：完全重复 2,568 行（0.48%）")

In [ ]:
quality.dq_record(con, "DQ-06", "唯一性", "完全重复记录", "ods_signal（全部原始字段）", f"{dup_mask.sum():,} 行（{dup_mask.mean():.2%}）", "警告",
          evidence="所有字段完全相同，疑似采集或入库时重复写入", decision="每组只保留一条")

In [ ]:
# 这一格在做什么：同一条记录里有两个时间戳，看它们差多少、该信哪个。
# 常见情形是一个「事件发生时间」、一个「记录写入时间」。差几十毫秒对分钟级的分析没影响，
# 但必须先量一量再下这个结论——不能假设。
lag = raw["timestamp"] - raw["timestamp1"]          # 单位：毫秒（两个原始值都是毫秒时间戳）
print(lag.describe(percentiles=[0.5, 0.9, 0.99]).round(0))
print(f"timestamp 早于 timestamp1 的：{(lag < 0).sum()} 行")

In [ ]:
quality.dq_record(con, "DQ-07", "一致性", "两个时间戳的差", "ods_signal.timestamp / timestamp1",
          f"中位 {lag.median():.0f} 毫秒，99% 在 {lag.quantile(0.99) / 1000:.1f} 秒内", "通过",
          evidence=f"只有 {(lag < 0).sum()} 行为负数", decision="统一用 timestamp 作为事件时间，差异对分钟级分析没有影响")

In [ ]:
from signal_mysql import database as mysql
# ════ 全天最重要的一格 ════
# 这一格在做什么：验证「IMSI 前 15 位能不能代表一个人」。
#
# 为什么会有这个问题：标准 IMSI 就是 15 位，这份数据却是 18 位。很自然的猜测是
# “前 15 位是人，后 3 位是某种会话编号”。听上去很合理——但这是**假设，不是事实**。
#
# 怎么验证：找一个**独立的**标识来当裁判。这里用手机号：如果前 15 位真的代表一个人，
# 那么同一个前 15 位下面不该出现两个不同的手机号。一查就知道成不成立。
#
# 讲台提示：**先让学生猜**“前 15 位能不能代表一个人”，举手表决，再跑这一格。
# 猜错的体验比直接给结论有用得多。
valid = raw.dropna(subset=["imsi"])
# 排除两类会污染结论的脏数据：格式异常的 IMSI、占位手机号 6666（36 行、14 个 IMSI）
valid = valid[valid["imsi"].str.fullmatch(r"\d{18}") & (valid["phone"] != "6666")]
# 只看「哪个 IMSI 配过哪个手机号」这组对应关系，同一对出现多少次不重要，所以先去重
pairs = valid[["imsi", "phone"]].drop_duplicates()
pairs["imsi15"] = pairs["imsi"].str[:15]                       # 截出前 15 位，作为“另一种认人方式”

per18 = pairs.groupby("imsi")["phone"].nunique()      # 每个 18 位 IMSI 对应几个手机号
per15 = pairs.groupby("imsi15")["phone"].nunique()    # 每个前 15 位对应几个手机号
key_check = pd.DataFrame({
    "grouping": ["按前 15 位", "按完整 18 位"],
    "n_ids": [len(per15), len(per18)],
    "n_multi_phone_ids": [(per15 > 1).sum(), (per18 > 1).sum()],
})
key_check["share"] = key_check["n_multi_phone_ids"] / key_check["n_ids"]
mysql.write_frame(key_check, "ads_key_check", con, if_exists="replace", index=False)
key_check

In [ ]:
multi = per15[per15 > 1].index
affected = pairs.loc[pairs["imsi15"].isin(multi), "imsi"].nunique()
suffix_kinds = pairs["imsi"].str[-3:].nunique()
print(f"对应多个手机号的前 15 位分组：{len(multi)} 组，里面装着 {affected:,} 个不同的 18 位 IMSI，"
      f"占全部用户的 {affected / len(per18):.1%}")
print(f"后 3 位一共有 {suffix_kinds} 种取值——如果真是“会话实例号”，应该是 001、002 这样的小编号")

In [ ]:
from signal_mysql import database as mysql
k = mysql.read_frame("SELECT * FROM ads_key_check", con)
rows = [(f"按前 15 位分组\n（{k.loc[0, 'n_ids']:,} 组）", k.loc[0, "n_multi_phone_ids"], k.loc[0, "n_ids"], "组对应多个手机号"),
        (f"被牵连的用户\n（{len(per18):,} 个 18 位 IMSI）", affected, len(per18), "个用户会被错误合并"),
        (f"按完整 18 位\n（{k.loc[1, 'n_ids']:,} 个）", k.loc[1, "n_multi_phone_ids"], k.loc[1, "n_ids"], "个对应多个手机号")]
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
for i, (lab, bad, tot, word) in enumerate(rows):
    s = bad / tot
    ax.barh(i, s, color=RED, height=0.56)                       # 红色 = 主键不成立的部分
    ax.barh(i, 1 - s, left=s, color=GRAY, height=0.56)
    if bad:
        ax.text(s / 2, i, f"{s:.1%}", ha="center", va="center", color="white", fontweight="bold", fontsize=11)
        ax.text(s + 0.012, i, f"{bad:,} {word}", va="center", color=INK, fontsize=10)
    else:
        ax.text(0.012, i, f"0 {word}：18 位 IMSI 与手机号一一对应", va="center", color=INK, fontsize=10)
ax.set_yticks(range(3), [r[0] for r in rows], fontsize=10, color=INK)
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xticks([]); ax.grid(False)
title = f"按前 15 位认人，{affected / len(per18):.0%} 的用户会被错误合并"
viz.finish(fig, title,
           "红色 = 主键不成立的部分；已排除 8 条格式异常的 IMSI 和占位手机号 6666；口径：ods_signal 全部日期，(IMSI, 手机号) 去重后",
           SOURCE, "D2_02_主键验证.png")

In [ ]:
# 对账
assert k.loc[0, "n_multi_phone_ids"] == 729 and k.loc[1, "n_multi_phone_ids"] == 0
assert affected == 1_707
print("✅ 对账通过：前 15 位有 729 组对应多个手机号，牵连 1,707 个用户；18 位 IMSI 与手机号一一对应")

In [ ]:
quality.dq_record(con, "DQ-08", "唯一性", "用户主键：IMSI 前 15 位能否代表一个人", "ods_signal.imsi",
          f"前 15 位有 {len(multi)} 组对应多个手机号（{len(multi) / len(per15):.1%}）", "不通过",
          evidence=f"这些分组牵连 {affected:,} 个 18 位 IMSI（{affected / len(per18):.1%}）；18 位 IMSI 与手机号一一对应",
          decision="用户主键改用完整 18 位 IMSI")

In [ ]:
from signal_mysql import database as mysql
# 这一格在做什么：**按确定好的顺序**执行清洗，并把每一步剔掉多少行记下来。
# 为什么要逐步记录：
#   ① 《数据质量报告》里的「清洗说明」就是这张表，不是事后回忆出来的；
#   ② 哪一步剔得异常多，一眼就能看见（这里是“不是 10 月 3 日”占了绝大部分）；
#   ③ 别人拿同一份原始数据，按同样的顺序能跑出同样的行数——这就是可复现。
steps = []
df = raw.drop(columns=["tmp0", "tmp1"])                        # 整列为空的两个字段，D2.1 已决定删除
n0 = len(df)                                                   # 清洗前的行数，后面所有占比的分母

def log_step(name, before, after):
    steps.append({"step": name, "rows_removed": before - after, "rows_after": after})

n = len(df); df = df.dropna(subset=["imsi", "lac_id", "cell_id"]);      log_step("缺 imsi / lac / cell", n, len(df))
n = len(df); df = df[df["imsi"].str.fullmatch(r"\d{18}")];              log_step("IMSI 格式异常", n, len(df))
n = len(df); df = df.drop_duplicates(subset=orig_cols);                  log_step("完全重复", n, len(df))
n = len(df); df = df[df["ts"].dt.strftime("%Y-%m-%d") == DAY];           log_step("不是 10 月 3 日", n, len(df))

steps = pd.DataFrame(steps)
steps["removed_share"] = steps["rows_removed"] / n0
assert steps["rows_removed"].sum() + len(df) == n0            # 核对数：剔除 + 剩余 = 原始
mysql.write_frame(steps, "ads_clean_steps", con, if_exists="replace", index=False)
steps

In [ ]:
from signal_mysql import database as mysql
dwd = pd.DataFrame({
    "imsi": df["imsi"],
    "ts": df["ts"],
    "dt": DAY,
    "lac_id": df["lac_id"].astype("int64"),
    "cell_id": df["cell_id"].astype("int64"),
    "nid": df["nid"],
})
# lac_id 和 cell_id 要合起来才能唯一定位一个扇区，拼成 "16789-67567924" 这样的单字段，
# D3 关联基站表时直接用它当连接键，比两个字段一起 join 简单也不易错。
dwd["lac_cell"] = dwd["lac_id"].astype(str) + "-" + dwd["cell_id"].astype(str)
# 按人、再按时间排好序。D6 的停留识别要求“同一个人的记录按时间相邻”，在这里排好，后面每天都省事。
dwd = dwd.sort_values(["imsi", "ts"]).reset_index(drop=True)
# ⚠️ 写库前的坑：ts 和原版一样存成固定格式的文本（MySQL 里是 TEXT 列）。
# 不统一格式的话，有的行带微秒、有的不带，读回来 pd.to_datetime 会解析出空值，
# 而且字符串比较大小会出错（"2018-10-03 9:05" > "2018-10-03 10:05"）。
# 所以这里统一成固定宽度、毫秒精度的文本。直接写 pandas 时间列的话，write_frame 会建成 DATETIME(3)，毫秒也不丢；
# 保留文本是为了和原版的 dwd_signal 逐行一致，后面各天读这张表的代码不用改。
dwd["ts"] = dwd["ts"].dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

mysql.write_frame(dwd, "dwd_signal", con, if_exists="replace", index=False)
con.commit()      # MySQL：写完马上提交。连接已在事务里时 pandas 不会替你提交，没提交的行关连接时会被回滚
n_db = mysql.read_frame("SELECT COUNT(*) AS n FROM dwd_signal", con)["n"][0]      # 写完读回来核对
print(f"dwd_signal：DataFrame {len(dwd):,} 行，库里 {n_db:,} 行，用户 {dwd['imsi'].nunique():,} 个")
dwd.head(3)

In [ ]:
# 对账
assert len(dwd) == n_db == 515_931 and dwd["imsi"].nunique() == 5_644
print("✅ 对账通过：dwd_signal 515,931 行，10 月 3 日用户 5,644 个")

In [ ]:
from signal_mysql import database as mysql
steps = mysql.read_frame("SELECT * FROM ads_clean_steps", con)
n1 = int(steps["rows_after"].iloc[-1]); removed = n0 - n1
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1.35, 1])
# 左：瀑布图，数值轴从 0 开始；红色 = 剔除项
labels = ["原始"] + ["缺 imsi/\nlac/cell", "IMSI\n格式异常", "完全\n重复", "不是\n10 月 3 日"] + ["清洗后"]
level = n0
a1.bar(0, n0 / 1e4, color=MAIN_DARK, width=0.62)
for i, kk in enumerate(steps["rows_removed"], start=1):
    a1.bar(i, kk / 1e4, bottom=(level - kk) / 1e4, color=RED, width=0.62)
    a1.plot([i - 0.31, i + 0.31], [level / 1e4] * 2, color=FAINT, lw=0.8, ls=":")
    a1.text(i, level / 1e4 + 0.8, f"−{kk:,}", ha="center", va="bottom", fontsize=9, color=RED)
    level -= kk
a1.bar(len(steps) + 1, n1 / 1e4, color=MAIN_DARK, width=0.62)
for xi, v in [(0, n0), (len(steps) + 1, n1)]:
    a1.text(xi, v / 1e4 + 3.2, f"{v:,}", ha="center", va="bottom", fontsize=9.5, color=INK, fontweight="bold")
a1.set_xticks(range(len(labels)), labels, fontsize=8.8); a1.set_ylim(0, 62); a1.set_ylabel("万行")
a1.set_title(f"总量：只去掉了 {removed / n0:.1%}", loc="left", fontsize=10.5, color=INK, fontweight="bold")
# 右：剔除的构成，最大一项强调
s_ = steps.set_index("step")["rows_removed"].sort_values()
b_ = a2.barh(s_.index, s_.values, color=[RED if v == s_.max() else GRAY for v in s_], height=0.6)
a2.bar_label(b_, labels=[f"{v:,}（{v / n0:.2%}）" if v / n0 >= 0.0001 else f"{v:,}（<0.01%）" for v in s_], padding=4,
             fontsize=9.5, color=INK)
a2.set_xlim(0, s_.max() * 1.75); a2.set_xticks([]); a2.grid(False); a2.tick_params(axis="y", labelsize=9.5, labelcolor=INK)
a2.set_title(f"剔除的 {removed:,} 行由什么构成", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"清洗共剔除 {removed / n0:.1%} 的记录，其中 {steps['rows_removed'].iloc[-1] / n0 * 100:.1f} 个百分点是“不是 10 月 3 日”"
viz.finish(fig, title,
           "按执行顺序剔除；左：瀑布图，数值轴从 0 开始，红色为剔除项；右：各步剔除行数，占比的分母是原始 532,828 行",
           SOURCE, "D2_03_清洗步骤.png")

In [ ]:
chk = mysql.connect(config.DB_SCHEMA)      # 新连接 = 另一个会话：只看得到已经提交的数据
try:
    for tbl in ["dq_check_result", "ads_missing_by_field", "ads_key_check", "ads_clean_steps", "dwd_signal"]:
        cnt = mysql.read_frame(f"SELECT COUNT(*) AS n FROM {tbl}", chk)["n"][0]
        print(f"{tbl:<22}{cnt:>9,}")
finally:
    chk.close()                            # 用完就关，连接还回连接池

In [ ]:
# ── 代码格 27 · 需求卡 2.8：用数据反推 nid 的含义 ──
from signal_mysql import database as mysql
# 这一格在做什么：不看文档，用数据本身反推 nid 这个字段是什么意思。
# 思路：如果某个 nid 代表「位置更新」类事件，那么它出现时，本人的位置区（LAC）
# 应该比平常更容易和上一条不一样。把各个 nid 的“伴随位置区变化的比例”排出来一看便知。
# 讲台提示：这一卡最适合讲“AI 或同事随口给的数据字典不能直接信，要用数据验一遍”。
x = dwd[["imsi", "ts", "lac_id", "nid"]].copy()      # dwd 在 2.6 已按 imsi、ts 排好序
# shift() 取“本人上一条记录”的值。必须先 groupby("imsi")，否则会跨人取到别人的记录。
prev_lac = x.groupby("imsi")["lac_id"].shift()
x["LAC变化"] = prev_lac.notna() & (x["lac_id"] != prev_lac)
x = x[prev_lac.notna()]                              # 每个人的第一条没有“上一条”，不参与比较
by_nid = x.groupby("nid").agg(n_records=("LAC变化", "size"), lac_change_share=("LAC变化", "mean"))
top12 = by_nid.nlargest(12, "n_records").sort_values("lac_change_share")
overall = x["LAC变化"].mean()
mysql.write_frame(top12, "ads_nid_lac_change", con, if_exists="replace")
fig, ax = plt.subplots(figsize=(10, 5.6))
best = top12["lac_change_share"].idxmax()
colors = [MAIN if i == best else GRAY for i in top12.index]
bars = ax.barh([f"nid={i}" for i in top12.index], top12["lac_change_share"], color=colors, height=0.66)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in top12["lac_change_share"]], padding=4, color=INK)
ax.axvline(overall, color=MUTED, lw=1, ls=(0, (4, 3)))
ax.text(overall, len(top12) - 0.4, f" 全部记录平均 {overall:.0%}", color=MUTED, fontsize=9, va="bottom")
ax.set_xlim(0, top12["lac_change_share"].max() * 1.2)
ax.set_xticks([])
ax.grid(False)
ax.spines["bottom"].set_visible(False)
title = f"nid={best} 有 {top12.loc[best, 'lac_change_share']:.0%} 伴随位置区变化，远高于平均的 {overall:.0%}"
viz.finish(fig, title,
           "每类事件发生时，位置区（LAC）与本人上一条记录不同的比例；取记录数最多的 12 个编码；口径：dwd_signal",
           SOURCE, "D2_04_nid与位置区变化.png")

In [ ]:
# ── 代码格 28：把字典里的说法调出来对照 ──
from signal_mysql import database as mysql
mapping = mysql.read_frame("SELECT * FROM ods_nid_mapping", con)
mapping[mapping["nid"].isin([best, 2])]

In [ ]:
# ── 代码格 29：把「字典不可信」记进稽核表 ──
quality.dq_record(con, "DQ-09", "准确性", "nid 字典与数据行为是否一致", "ods_nid_mapping",
          f"nid={best} 有 {top12.loc[best, 'lac_change_share']:.0%} 伴随位置区变化，字典却标为会话类事件", "不通过",
          evidence=f"全部记录平均 {overall:.0%}；字典标为“位置更新”的 nid=2 只有 {by_nid.loc[2, 'lac_change_share']:.0%}",
          decision="字典不采用；nid 含义标为未知，向数据提供方索取接口文档")

In [ ]:
# ── 代码格 30 · 需求卡 2.9：两个时间戳的差画成分布 ──
# 只看平均值会被极端值带偏，看分布更稳。累计分布曲线（ECDF）直接回答“差值在多少以内的记录占多少”。
d_ = lag.dropna(); nonpos = int((d_ <= 0).sum()); d_ = d_[d_ > 0]
med, p99 = d_.median(), d_.quantile(0.99)
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
ax.ecdf(d_, color=MAIN, lw=2)
ax.set_xscale("log"); ax.set_xlim(1, d_.max() * 1.1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="both")
ax.plot([med, med], [0, 0.5], color=MUTED, lw=1, ls="--"); ax.plot(med, 0.5, "o", color=MAIN, ms=5)
ax.text(med / 1.25, 0.54, f"中位数 {med:,.0f} 毫秒", ha="right", va="bottom", fontsize=9.5, color=INK)
ax.plot([p99, p99], [0, 0.99], color=MUTED, lw=1, ls="--"); ax.plot(p99, 0.99, "o", color=MAIN, ms=5)
ax.text(p99 * 1.15, 0.94, f"99 分位 {p99:,.0f} 毫秒", va="top", fontsize=9.5, color=INK)
ax.set_xlabel("timestamp − timestamp1（毫秒，对数刻度）")   # 真正的负号：靠 DejaVu Sans 兜底才不会变方框
ax.set_ylabel("差值不超过横轴数值的记录占比")
title = f"两个时间戳相差中位 {med:,.0f} 毫秒，99% 在 {p99 / 1000:.1f} 秒以内"
viz.finish(fig, title,
           f"累计分布（ECDF）；不含 {nonpos:,} 条差值 ≤ 0 的记录；口径：ods_signal 全部 532,828 行",
           SOURCE, "D2_05_时间戳差分布.png")

In [ ]:
# ── 代码格 31 · 需求卡 2.10：信令多久来一条（观测稀疏性）──
from signal_mysql import database as mysql
# 这一格在做什么：量一量「信令多久来一条」——也就是我们对一个人的观测有多稀疏。
# 这件事决定了后面所有结论的精度上限：信令是**网络事件触发**的（打电话、发消息、切基站……），
# 不是每隔几分钟定时采样。所以**“没有信令”不等于“人没动”**，只等于“这段时间我们看不见他”。
s = mysql.read_frame("SELECT imsi, ts FROM dwd_signal", con, parse_dates=["ts"]).sort_values(["imsi", "ts"])
# diff() 算相邻两条的时间差；同样要先 groupby("imsi")，否则会算到上一个人的最后一条
gap = (s.groupby("imsi")["ts"].diff().dt.total_seconds() / 60).dropna()   # 单位：分钟
q50, q95 = gap.quantile(0.5), gap.quantile(0.95)
obs = (s.assign(h=s["ts"].dt.hour).groupby("h")["imsi"].nunique()
         .reindex(range(24), fill_value=0) / s["imsi"].nunique())         # 每小时至少有一条信令的用户占比
hmin = int(obs.idxmin())
print(f"间隔中位 {q50:.1f} 分钟，95 分位 {q95:.0f} 分钟；{hmin} 点只有 {obs[hmin]:.0%} 的人有信令，最多的小时 {obs.max():.0%}")

In [ ]:
# ── 代码格 32 · 需求卡 2.10：画 D2_06「观测稀疏性」──
g_ = gap[gap > 0]
lo_h, hi_h = obs.idxmin(), obs.idxmax()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1, 1.05])
a1.ecdf(g_, color=MAIN, lw=2)
a1.set_xscale("log"); a1.set_xlim(0.05, 1000)
a1.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
a1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
a1.grid(axis="both")
for xv, yv, lab in [(q50, 0.5, f"中位 {q50:.0f} 分钟"), (q95, 0.95, f"95 分位 {q95:.0f} 分钟")]:
    a1.plot([xv, xv], [0, yv], color=MUTED, lw=1, ls="--"); a1.plot(xv, yv, "o", color=MAIN, ms=5)
    a1.text(xv * 1.3, yv - 0.07, lab, fontsize=9.5, color=INK)
a1.set_xlabel("同一用户相邻两条信令的间隔（分钟，对数刻度）")
a1.set_title("相邻信令的时间间隔", loc="left", fontsize=10.5, color=INK, fontweight="bold")
# 右：时间序列用折线，不用 24 根柱；只标最低点（红 = 警示）和最高点
a2.plot(obs.index, obs.values, color=GRAY_DARK, lw=2, marker="o", ms=3.5)
for hh, c_ in [(lo_h, RED), (hi_h, MAIN)]:
    a2.plot(hh, obs[hh], "o", color=c_, ms=7, zorder=5)
    a2.text(hh, obs[hh] + (-0.07 if c_ == RED else 0.04), f"{hh} 点 {obs[hh]:.0%}", ha="center", fontsize=9.5,
            color=c_, fontweight="bold")
a2.set_ylim(0, 1); a2.set_xlim(-0.5, 23.5)
a2.set_xticks(range(0, 24, 3), [f"{x} 点" for x in range(0, 24, 3)])
a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
a2.set_title("每小时至少有一条信令的用户占比", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"相邻信令中位间隔 {q50:.0f} 分钟，但 5% 超过 {q95:.0f} 分钟；凌晨 {lo_h} 点只有 {obs[lo_h]:.0%} 的人有信令"
viz.finish(fig, title,
           "信令是网络事件触发的，不是定时采样：没有信令不代表人没动；口径：dwd_signal（10-03，5,644 人）",
           SOURCE, "D2_06_观测稀疏性.png")

In [ ]:
# ── 代码格 33 · 收尾：关闭数据库连接 ──
con.close()